<a href="https://colab.research.google.com/github/1rishu0/Computer-Vision/blob/main/HOMERWORK_Computer_Vision_Masterclass_Neural_networks_for_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural network for image classification - cats and dogs

## Approach 1 - extract all the pixels

### Importing the libraries

In [ ]:
import cv2
import numpy as np
import os
import zipfile
from google.colab.patches import cv2_imshow
import tensorflow as tf
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
tf.__version__

### Extracting pixels from images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = '/content/drive/MyDrive/Cursos - recursos/Computer Vision Masterclass/Datasets/cat_dog_1.zip'
zip_object = zipfile.ZipFile(file = path, mode = 'r')
zip_object.extractall('./')
zip_object.close()

In [ ]:
directory_train = '/content/cat_dog_1/train'
directory_test = '/content/cat_dog_1/test'
files_train = [os.path.join(directory_train, f) for f in sorted(os.listdir(directory_train))]
files_test = [os.path.join(directory_test, f) for f in sorted(os.listdir(directory_test))]

In [ ]:
height, width = 128, 128

#### Train

In [ ]:
images = []
classes = []
for image_path in files_train:
  try:
    image = cv2.imread(image_path)
    (H, W) = image.shape[:2]
  except:
    continue

  image = cv2.resize(image, (width, height))
  image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

  cv2_imshow(image)

  image = image.ravel()
  print(image.shape)

  images.append(image)

  image_name = os.path.basename(os.path.normpath(image_path))
  if image_name.startswith('c'):
    # Cat
    class_name = 0
  else:
    # Dog
    class_name = 1

  classes.append(class_name)
  print(class_name)

In [ ]:
X_train = np.asarray(images)
y_train = np.asarray(classes)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

#### Test

In [ ]:
images = []
classes = []
for image_path in files_test:
  try:
    image = cv2.imread(image_path)
    (H, W) = image.shape[:2]
  except:
    continue

  image = cv2.resize(image, (width, height))
  image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

  cv2_imshow(image)

  image = image.ravel()
  print(image.shape)

  images.append(image)

  image_name = os.path.basename(os.path.normpath(image_path))
  if image_name.startswith('c'):
    # Cat
    class_name = 0
  else:
    # Dog
    class_name = 1

  classes.append(class_name)
  print(class_name)

In [ ]:
X_test = np.asarray(images)
y_test = np.asarray(classes)

In [ ]:
X_test.shape, y_test.shape

In [ ]:
sns.countplot(y_train);

In [ ]:
sns.countplot(y_test);

### Normalizing the data

In [ ]:
X_train[0].max(), X_train[0].min()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train[0].max(), X_train[0].min()

In [ ]:
X_test[0].max(), X_test[0].min()

### Building and training the neural network

In [ ]:
(16384 + 2) / 2

In [ ]:
# 16384 -> 8193 -> 8193 -> 1
network1 = tf.keras.models.Sequential()
network1.add(tf.keras.layers.Dense(input_shape=(16384,), units=8193, activation='relu'))
network1.add(tf.keras.layers.Dense(units=8193, activation = 'relu'))
network1.add(tf.keras.layers.Dense(units = 1, activation = 'sigmoid'))

In [ ]:
network1.summary()

In [ ]:
# https://keras.io/api/optimizers/
# https://keras.io/api/losses/
network1.compile(optimizer='Adam', loss='binary_crossentropy', metrics = ['accuracy'])

In [ ]:
history = network1.fit(X_train, y_train, epochs=10)

### Evaluating the neural network

In [ ]:
plt.plot(history.history['loss']);

In [ ]:
plt.plot(history.history['accuracy']);

In [ ]:
X_test.shape

In [ ]:
predictions = network1.predict(X_test)
predictions

In [ ]:
# 0 False - Bart
# 1 True  - Homer

In [ ]:
predictions = (predictions > 0.5)
predictions

In [ ]:
y_test

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, predictions)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, predictions)
cm

In [ ]:
sns.heatmap(cm, annot=True);

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions))